# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub


In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [4]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Connected!")

Connected!


## 1. Unit of analysis + time window

**Unit of analysis:** One pseudonymized content page at a single decision point.

**Source table:** `fact_content_daily_performance`

**Time window:** March 2026 (`month=2026-03`)

**Task:** Rank content pages for refresh review.

**Output:** A ranked review queue for the SEO/content team.

I use March 2026 because it is a mid-panel month. I avoid using the June 2026 sample to develop labels because the final month should remain a future evaluation period.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

## 2. Fields: feature / label / context / excluded

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- content_visible_query_count
- rare_impressions_share

These are available before making the refresh decision.

### Label / Proxy
Declining page (temporary proxy based on later observed impressions).

### Context
- client_hash_id
- content_hash_id
- report_date

Used only for grouping, joins, and validation.

### Excluded
- IDs as model features.
- Future information.
- Any field used to create the label.
- Product decision flags.

These are excluded to prevent leakage.


In [6]:
print("Features")
print([
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "content_visible_query_count",
    "rare_impressions_share"
])

print("\nLabel")
print("Declining page")

print("\nContext")
print([
    "client_hash_id",
    "content_hash_id",
    "report_date"
])

print("\nExcluded")
print([
    "Future outcome columns",
    "IDs",
    "Label-derived columns"
])

Features
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'content_visible_query_count', 'rare_impressions_share']

Label
Declining page

Context
['client_hash_id', 'content_hash_id', 'report_date']

Excluded
['Future outcome columns', 'IDs', 'Label-derived columns']


## 3. Verify it with queries

The following three queries verify:

1. Grain
2. Row count and date range
3. Data availability

In [7]:
# Query 1
grain = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) c
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*)>1
LIMIT 5
""").df()

grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [9]:
counts = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

counts

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [11]:
# Query 3
availability = con.sql(f"""
SELECT
COUNT(*) total_rows,
COUNT(*) FILTER(
WHERE ga4_data_available IS TRUE
) available_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows
0,9841378,413966


In [12]:
features = con.sql(f"""
SELECT
content_hash_id,
SUM(gsc_impressions) impressions,
SUM(gsc_clicks) clicks,
AVG(gsc_avg_position) avg_position
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY content_hash_id
LIMIT 10
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions,clicks,avg_position
0,content_7a105f548d9c6916,6523.0,7.0,7.209549
1,content_a3ea9792f793ec72,453.0,0.0,2.987198
2,content_36c36abc7650d7af,5630.0,6.0,6.724039
3,content_a7da352b73b02668,4944.0,13.0,7.244844
4,content_f39be42b42a4e8f6,42.0,0.0,14.432540


In [13]:
print("Honest features:")
print([
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
])

print("\nLeaky feature example:")
print("imp_last30")

print("\nRemoving leaky feature to avoid leakage.")

Honest features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']

Leaky feature example:
imp_last30

Removing leaky feature to avoid leakage.


## 4. Data limits

- Client history is unbalanced.
- Some GA4 values are unavailable.
- Early rows may contain only GSC data.
- This dataset supports decision making but cannot prove causation.
- Patterns observed in March 2026 may not generalize to all time periods.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.